# Ollama-compatible inference on Kaggle (Flask + ngrok)

This notebook runs your **base model + LoRA adapter** on a Kaggle GPU and exposes a minimal HTTP API that matches what the AI Legal Advisor backend expects from **Ollama**:

- `GET /api/tags` — used by `/health` on the laptop backend
- `POST /api/generate` — same JSON shape as Ollama (`prompt`, `options.num_predict`, etc.; response field `response`)

**Before you run**

1. **Accelerator**: set the notebook to **GPU** (T4 or better; 8B + LoRA needs enough VRAM).
2. **Adapter**: upload a Kaggle **Dataset** whose root (or a subfolder) contains **`adapter_config.json`** plus the LoRA weights. The mount path is always **`/kaggle/input/<dataset-slug>/`** (slug = dataset name, with hyphens). If your files sit in the dataset root, set `ADAPTER_DIR` to `/kaggle/input/<slug>` — not `/slug/model` unless you actually have a `model` subfolder. Run the path check cell if unsure.
3. **Hugging Face**: for gated Llama weights, add a Kaggle secret `HF_TOKEN` (and enable it for this notebook).
4. **ngrok**: add a Kaggle secret `NGROK_AUTHTOKEN` from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).
5. **Backend `.env`**: set `OLLAMA_URL` to the printed **base** URL only, e.g. `https://....ngrok-free.app` — do **not** append `/api/tags` or `/api/generate` (the Flask backend adds those). Match `OLLAMA_MODEL` to `MODEL_ALIAS` in this notebook.

**Browser check**: opening only the base URL may show an empty page or old behavior; use **`/api/tags`** (e.g. `https://....ngrok-free.app/api/tags`) to confirm the server is up. The AI Legal Advisor backend only calls `/api/tags` and `/api/generate`, not `/`.

**Timeouts**: the server cell **preloads** the model before ngrok starts so your laptop is less likely to hit HTTP read timeouts. On the laptop, set **`OLLAMA_REQUEST_TIMEOUT=300`** (or **600** if needed) in `.env` — the first remote generate can be slow.

**Security**: the tunnel is public. Use a fresh ngrok URL for demos; do not commit tokens.


In [ ]:
%pip install -q flask pyngrok "transformers>=4.45.0" "accelerate>=0.33.0" "peft>=0.12.0" torch

In [ ]:
import os

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        if not os.environ.get("HF_TOKEN"):
            os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
        if not os.environ.get("NGROK_AUTHTOKEN"):
            os.environ["NGROK_AUTHTOKEN"] = secrets.get_secret("NGROK_AUTHTOKEN")
    except Exception as e:
        print("Kaggle secrets (HF_TOKEN / NGROK_AUTHTOKEN):", e)

In [ ]:
# Optional: see what Kaggle mounted under /kaggle/input (find your dataset slug and layout)
from pathlib import Path

root = Path("/kaggle/input")
if root.is_dir():
    for d in sorted(root.iterdir()):
        print(d)
        if d.is_dir():
            for p in sorted(d.rglob("adapter_config.json")):
                print("  ->", p.parent)
else:
    print("Not on Kaggle or /kaggle/input missing — run this on Kaggle after adding your Dataset.")

In [ ]:
import os
import threading
from pathlib import Path

import torch
from flask import Flask, jsonify, request
from huggingface_hub import login
from peft import PeftModel
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer


def resolve_local_adapter_dir(preferred: str) -> str:
    """PEFT treats missing paths like Hub repo ids — require a real adapter_config.json on disk."""
    p = Path(preferred).expanduser()
    if (p / "adapter_config.json").is_file():
        return str(p.resolve())
    if p.parent != p and (p.parent / "adapter_config.json").is_file():
        return str(p.parent.resolve())
    kaggle_in = Path("/kaggle/input")
    if kaggle_in.is_dir():
        for child in sorted(kaggle_in.iterdir()):
            if not child.is_dir():
                continue
            for candidate in (
                child,
                child / "model",
                child / "adapter",
                child / "lora",
                child / "backend" / "model",
            ):
                if candidate.is_dir() and (candidate / "adapter_config.json").is_file():
                    return str(candidate.resolve())
    raise FileNotFoundError(
        f"No adapter_config.json under '{preferred}'. "
        "Run the cell above to list /kaggle/input, then set ADAPTER_DIR to the directory that contains adapter_config.json "
        "(often /kaggle/input/<dataset-slug> if you uploaded files at the dataset root)."
    )


# --- set to the folder that contains adapter_config.json (or use KAGGLE_ADAPTER_DIR env) ---
ADAPTER_DIR = os.environ.get("KAGGLE_ADAPTER_DIR", "/kaggle/input/adapter-dataset/model")
ADAPTER_DIR = resolve_local_adapter_dir(ADAPTER_DIR)
print("Resolved ADAPTER_DIR:", ADAPTER_DIR)

BASE_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
MODEL_ALIAS = "llama3.1:8b"
SERVE_PORT = 5000

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

app = Flask(__name__)

tokenizer = None
model = None


@app.get("/")
def root():
    """So opening the ngrok base URL in a browser is not a bare 404."""
    return jsonify(
        {
            "service": "ollama-compatible-shim",
            "ok": True,
            "try": {"health_models": "/api/tags", "generate": "POST /api/generate"},
            "model_alias": MODEL_ALIAS,
        }
    )


def model_device():
    return next(model.parameters()).device


def load_model():
    global tokenizer, model
    if model is not None:
        return
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, ADAPTER_DIR, local_files_only=True)
    model.eval()


@app.get("/api/tags")
def tags():
    return jsonify(
        {"models": [{"name": MODEL_ALIAS, "size": 0, "digest": "", "modified_at": ""}]}
    )


@app.post("/api/generate")
def generate():
    load_model()
    body = request.get_json(force=True, silent=True) or {}
    prompt = body.get("prompt", "")
    opts = body.get("options") or {}
    max_new = int(opts.get("num_predict", 512))
    temperature = float(opts.get("temperature", 0.3))
    top_p = float(opts.get("top_p", 0.9))

    inputs = tokenizer(prompt, return_tensors="pt").to(model_device())
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0][inputs["input_ids"].shape[-1] :]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return jsonify({"response": text, "done": True, "model": body.get("model", MODEL_ALIAS)})


def run_server():
    app.run(host="0.0.0.0", port=SERVE_PORT, threaded=True)


print("Preloading model on GPU (can take several minutes; avoids client timeouts on first /api/generate)...")
load_model()
print("Model ready. Starting Flask + ngrok.")

threading.Thread(target=run_server, daemon=True).start()

if not os.environ.get("NGROK_AUTHTOKEN"):
    raise RuntimeError("Set NGROK_AUTHTOKEN (Kaggle secret or env var).")

ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
public = ngrok.connect(SERVE_PORT)
print("Public URL (set as OLLAMA_URL on backend):", public.public_url)
print("OLLAMA_MODEL on backend must match MODEL_ALIAS:", MODEL_ALIAS)
